In [1]:
import pandas as pd
import numpy as np
import random

In [ ]:
from typing import Callable

def accuracy(y_true, y_pred) -> float:
    return np.mean(y_true == y_pred)

def precision(y_true, y_pred) -> float:
    mask = y_pred == 1
    return y_true[mask].mean()

def recall(y_true, y_pred) -> float:
    mask = y_true == 1
    return y_pred[mask].mean()

def f1(y_true, y_pred) -> float:
    precision_ = precision(y_true, y_pred)
    recall_ = recall(y_true, y_pred)
    f1 = 2 * precision_ * recall_ / (precision_ + recall_)
    return f1

def roc_auc(y_true: pd.Series, y_pred: pd.Series):
    y_true = y_true.copy()
    y_true.name = 'true'
    y_pred = y_pred.copy()
    y_pred.name = 'pred'
    y_pred = np.round(y_pred, 10)
    
    y = pd.concat([y_pred, y_true], axis=1).sort_values(["pred", "true"], ascending=False)    

    roc_auc = 0 
    for i in range(len(y)):
        if y['true'].iloc[i] == 0:
            p = y['pred'].iloc[i]
            for j in range(i):
                if y['true'].iloc[j] == 1 and y['pred'].iloc[j] > p :
                    roc_auc += 1
                elif y['true'].iloc[j] == 1 and y['pred'].iloc[j] == p:
                    roc_auc += 0.5
    roc_auc /= np.sum(y['true']) * (len(y['true']) - np.sum(y['true']))

    return roc_auc


class MyLogReg:
    def __init__(self, n_iter: int = 10, learning_rate: float = 0.1,
                metric = None, reg = None, l1_coef = 0, l2_coef = 0,
                sgd_sample=None, random_state=42
                ):
        self.n_iter = n_iter
        self.learning_rate = learning_rate
        self.metric = metric
        self.reg = reg
        self.l1_coef = l1_coef
        self.l2_coef = l2_coef

        self.weights = None
        self.eps = 1e-15
        self.__metric_value = None
        self.sgd_sample = sgd_sample
        self._seed(random_state)

    def _seed(self, random_state):
        self.random_state = random_state 
        random.seed(self.random_state)

    def __str__(self):
        return f"MyLogReg class: n_iter={self.n_iter}, learning_rate={self.learning_rate}"

    def _accuracy(self, y_true, y_pred) -> float:
        return np.mean(y_true == y_pred)

    def _precision(self, y_true, y_pred) -> float:
        mask = y_pred == 1
        return y_true[mask].mean()

    def _recall(self, y_true, y_pred) -> float:
        mask = y_true == 1
        return y_pred[mask].mean()

    def _f1(self, y_true, y_pred) -> float:
        precision = self._precision(y_true, y_pred)
        recall = self._recall(y_true, y_pred)
        f1 = 2 * precision * recall / (precision + recall)
        return f1

    def _roc_auc(self, y_true: pd.Series, y_pred: pd.Series):
        y_true = y_true.copy()
        y_true.name = 'true'
        y_pred = y_pred.copy()
        y_pred.name = 'pred'
        y_pred = np.round(y_pred, 10)
        
        y = pd.concat([y_pred, y_true], axis=1).sort_values(["pred", "true"], ascending=False)    

        roc_auc = 0 
        for i in range(len(y)):
            if y['true'].iloc[i] == 0:
                p = y['pred'].iloc[i]
                for j in range(i):
                    if y['true'].iloc[j] == 1 and y['pred'].iloc[j] > p :
                        roc_auc += 1
                    elif y['true'].iloc[j] == 1 and y['pred'].iloc[j] == p:
                        roc_auc += 0.5
        roc_auc /= np.sum(y['true']) * (len(y['true']) - np.sum(y['true']))

        return roc_auc
    
    def log_loss(self, y_true, y_pred):
        return - np.mean(y_true * np.log(y_pred + self.eps) +  (1 - y_true) * np.log(1 - y_pred + self.eps)) 

    def _get_metric_value(self, y_true, y_pred):
        if self.metric != 'roc_auc':
            y_pred = y_pred > 0.5
        if self.metric == 'accuracy':
            return self._accuracy(y_true, y_pred)
        if self.metric == 'precision':
            return self._precision(y_true, y_pred)
        if self.metric == 'recall':
            return self._recall(y_true, y_pred)
        if self.metric == 'f1':
            return self._f1(y_true, y_pred)
        if self.metric == 'roc_auc':
            return self._roc_auc(y_true, y_pred)
    
    def fit(self, X: pd.DataFrame, y: pd.Series, verbose: int = 10):
        
        if '__bias' in X.columns:
            raise Exception("Column '__bias' exists in X")
        
        X = X.copy()

        X.insert(loc=0, column='__bias', value=1)

        count_features = len(X.columns)
        n = len(X)
            
        self.weights = pd.Series(1, X.columns)

        for i in range(1, self.n_iter + 1):
            
            y_pred = 1 / (1 + np.exp(-X.dot(self.weights)))
            log_loss = self.log_loss(y, y_pred)

            if self.reg == 'l1':
                log_loss += self.l1_coef * np.sum(np.abs(self.weights))

            if self.reg == 'l2':
                log_loss += self.l2_coef * np.sum(np.square(self.weights))

            if self.reg == 'elasticnet':
                log_loss += self.l1_coef * np.sum(np.abs(self.weights)) + self.l2_coef * np.sum(np.square(self.weights))
            
            if self.sgd_sample is not None:
                if isinstance(self.sgd_sample, int):
                    sgd_sample = self.sgd_sample
                elif isinstance(self.sgd_sample, float):
                    sgd_sample = round(self.sgd_sample * n)
                    
                sample_rows_idx = random.sample(range(X.shape[0]), sgd_sample)
            
                X_hat = []
                y_hat = []
                y_pred_hat = []
                indexes_hat = []
                for j in sample_rows_idx:
                    X_hat.append(X.iloc[j])
                    y_hat.append(y.iloc[j])
                    y_pred_hat.append(y_pred.iloc[j])
                    indexes_hat.append(X.index[j])
                X_hat = pd.concat(X_hat, axis=1).T
                y_hat = pd.Series(y_hat, index= indexes_hat)
                y_pred_hat = pd.Series(y_pred_hat, index= indexes_hat)
            else:
                X_hat = X
                y_hat = y
                y_pred_hat = y_pred
            

            gradient = (y_pred_hat - y_hat).dot(X_hat) / len(X_hat)

            if self.reg == 'l1':
                gradient += self.l1_coef * np.sign(self.weights)

            if self.reg == 'l2':
                gradient += self.l2_coef * self.weights * 2

            if self.reg == 'elasticnet':
                gradient += self.l1_coef * np.sign(self.weights) + self.l2_coef * self.weights * 2            
            
            if isinstance(self.learning_rate, float):
                learning_rate = self.learning_rate
            
            if isinstance(self.learning_rate, Callable):
                learning_rate = self.learning_rate(i)
            

            self.weights -= learning_rate * gradient

        y_pred = 1 / (1 + np.exp(-X.dot(self.weights)))
        self.__metric_value = self._get_metric_value(y, y_pred)

    def get_coef(self) -> np.ndarray:
        if self.weights is None:
            return None

        return self.weights.values[1:]

    def predict_proba(self, X: pd.DataFrame) -> pd.Series:
        X = X.copy()

        X.insert(loc=0, column='__bias', value=1)

        return 1 / (1 + np.exp(-X.dot(self.weights)))

    def predict(self, X: pd.DataFrame) -> pd.Series:
        y_pred = self.predict_proba(X)
        return y_pred > 0.5

    def get_best_score(self):
        return self.__metric_value


def get_entropy(y: pd.Series):
    unique_labels = y.unique()
    
    if len(unique_labels) == 1:
        return 0
    
    n = len(y)
    count_1 = y.sum()
    count_0 = n - count_1

    proba_0 = count_0 / n
    proba_1 = count_1 / n

    return - (np.log2(proba_0) * proba_0 + np.log2(proba_1) * proba_1)

def get_gini(y: pd.Series):
    unique_labels = y.unique()

    if len (unique_labels) == 1:
        return 0

    n = len(y)
    count_1 = y.sum()
    count_0 = n - count_1

    proba_0 = count_0 / n
    proba_1 = count_1 / n

    return 1 - proba_0 ** 2 - proba_1 ** 2
 
class Leaf:
    def __init__(self, parent, samples, proba, depth):
        self.parent = parent
        self.samples = samples
        self.proba = proba
        self.depth = depth

    def __str__(self):
        return f'{"  " * self.depth} {self.proba}'

class Node:
    def __init__(self, parent=None):
        self.parent = parent
        self.left = None
        self.right = None
        self.samples = None
        self.depth = 1
        self.split_value = None
        self.column = None
        self.ig = None
        self.is_left = None

    def __str__(self):
        return  '\n'.join([
        f'{"  " * self.depth} {self.column} {self.split_value}',
        f'{self.left}',
        f'{self.right}'])

    def predict_proba(self, row: pd.Series):
        if row.loc[self.column] <= self.split_value:
            if isinstance(self.left, Leaf):
                return self.left.proba
            else: 
                return self.left.predict_proba(row)
        else:
            if isinstance(self.right, Leaf):
                return self.right.proba
            else: 
                return self.right.predict_proba(row)

class MyTreeClf:
    def __init__(self, max_depth: int = 5, min_samples_split: int = 2,
                 max_leafs: int = 20, bins=None, criterion="entropy"):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_leafs = max_leafs
        self.nodes_cnt = 1
        self.leafs_cnt = None
        self.bins = bins
        self.split_values = None
        self.criterion = criterion
        self.fi = dict()
        self.N = None

    def __str__(self):
        return f"MyTreeClf class: max_depth={self.max_depth}, min_samples_split={self.min_samples_split}, max_leafs={self.max_leafs}"

    def __build_split_values(self, X):
        self.split_values = dict()
        for column in X.columns:
            unique_values = X[column].unique()
            if (len(unique_values) - 1 <= self.bins - 1):
                sorted_unique_values = np.sort(unique_values)
                self.split_values[column] = (sorted_unique_values[:-1] + sorted_unique_values[1:]) / 2
            else:
                count_elements, split_values = np.histogram(X[column], bins = self.bins)
                self.split_values[column] = split_values[1:-1]

    def get_split_values(self, X, column):
        if self.bins is None:
            unique_values = X[column].unique()
            sorted_unique_values = np.sort(unique_values)
            return (sorted_unique_values[:-1] + sorted_unique_values[1:]) / 2
        else:
            max_value = X[column].max()
            min_value = X[column].min()
            return self.split_values[column][ (min_value <= self.split_values[column]) & (max_value > self.split_values[column])]

    def __init_fi(self, X):
        for column in X.columns:
            self.fi[column] = 0 
    
    def fit(self, X: pd.DataFrame, y: pd.Series):
        self.root = Node()
        self.__init_fi(X)
        self.N = len(y)
        if self.bins is not None:
            self.__build_split_values(X)
            
        self.build_tree_recursive(self.root, X, y)
        self.leafs_cnt = self.nodes_cnt + 1

    def get_best_split(self, X: pd. DataFrame, y : pd.Series):

        igs = []
        
        for column in X.columns:
            split_values = self.get_split_values(X, column)
            if self.criterion == "entropy":
                criterion_function = get_entropy
            elif self.criterion == "gini":
                criterion_function = get_gini
            criterion = criterion_function(y)
            
            for split_value in split_values: 

                left_indexes = X.index[X[column] <= split_value]
                left_y = y.loc[left_indexes]
                left_criterion = criterion_function(left_y)
                
                right_indexes = X.index[X[column] > split_value]
                right_y = y.loc[right_indexes]
                right_criterion = criterion_function(right_y)
    
                ig = criterion - left_criterion * len(left_indexes) / len(y) - right_criterion * len(right_indexes) / len(y) 
    
                igs.append((column, split_value, ig))
        if igs:    
            return max(igs, key= lambda x: x[2])
        else:
            return None, None, None

    
    def predict_proba(self, X: pd.DataFrame) -> pd.Series:
        probas = []
        for index, row in X.iterrows():
            proba = self.root.predict_proba(row)
            probas.append(proba)
        return pd.Series(probas, X.index)

    def predict(self, X: pd.DataFrame) -> pd.Series:
        probas = self.predict_proba(X)
        return probas > 0.5
        
    def build_tree_recursive(self, node: Node, X, y):

        node.column, node.split_value, node.ig = self.get_best_split(X, y)
        
        if node.column is not None:
            self.fi[node.column] += len(y) / self.N * node.ig
            
            left_indexes = X.index[X[node.column] <= node.split_value]
            left_cnt = len(left_indexes)
            left_y = y.loc[left_indexes]
            
            if len(left_y.unique()) == 1 or left_cnt < self.min_samples_split or \
                self.nodes_cnt + 1 >= self.max_leafs or node.depth == self.max_depth \
                :
                node.left = Leaf(node, len(left_y), left_y.mean(), node.depth + 1)
            else:
                node.left = Node(node)
                node.left.depth = node.depth + 1
                node.left.is_left = True
                self.nodes_cnt += 1
                self.build_tree_recursive(node.left, X.loc[left_indexes], left_y)
    
            right_indexes = X.index[X[node.column] > node.split_value]
            right_cnt = len(right_indexes)
            right_y = y.loc[right_indexes]
            
            if len(right_y.unique()) == 1  or right_cnt < self.min_samples_split  or \
                self.nodes_cnt + 1 >= self.max_leafs or node.depth == self.max_depth:
                node.right = Leaf(node, len(right_y), right_y.mean(), node.depth + 1)
            else:
                node.right = Node(node)
                node.right.depth = node.depth + 1
                node.right.is_left = False
                self.nodes_cnt += 1
                self.build_tree_recursive(node.right, X.loc[right_indexes], right_y)
        else:
            parent = node.parent
            if node.is_left:
                parent.left = Leaf(parent, len(y), y.mean(), parent.depth + 1)
            else:
                parent.right = Leaf(parent, len(y), y.mean(), parent.depth + 1)
            self.nodes_cnt -= 1


class MyKNNClf:
    def __init__(self, k = 3, metric = 'euclidean', weight = 'uniform'):
        self.k = k 
        self.train_size = None
        self.metric = metric
        self.weight = weight

    def __str__(self):
        return f"MyKNNClf class: k={self.k}"

    def fit(self, X: pd.DataFrame, y: pd.Series):
        self.X = X.copy()
        self.y = y.copy()
        self.train_size = self.X.shape

    def _get_distances(self, x: pd.Series):
        if self.metric == 'euclidean':
            return np.sqrt(((self.X - x) ** 2).sum(axis=1))
        if self.metric == 'chebyshev':
            return (self.X - x).abs().max(axis=1)
        if self.metric == 'manhattan':
            return (self.X - x).abs().sum(axis=1)
        if self.metric == 'cosine':
            return 1 - (self.X * x).sum(axis=1) / ( np.sqrt((self.X ** 2).sum(axis=1)) * ((x ** 2).sum() ** 0.5) )

    def predict_proba(self, X: pd.DataFrame):
        y_pred = []
        for index, row in X.iterrows():
            distances = self._get_distances(row)
            distances.sort_values(ascending=True, inplace=True)
            train_indexes = distances.index[:self.k]
            k_nearest_neighbor_distances = distances.loc[train_indexes]
            k_nearest_neighbor_y = self.y.loc[train_indexes]
            if self.weight == 'uniform':
                proba = k_nearest_neighbor_y.mean()
            if self.weight == 'rank':
                k_nearest_neighbor_rank = pd.Series(range(1, self.k + 1), train_indexes)
                sum_weight = (1 / k_nearest_neighbor_rank).sum()
                mask = k_nearest_neighbor_y == 1
                proba = (1 / k_nearest_neighbor_rank[mask]).sum() / sum_weight
            if self.weight == 'distance':
                sum_weight = (1 / k_nearest_neighbor_distances).sum()
                mask = k_nearest_neighbor_y == 1
                proba = (1 / k_nearest_neighbor_distances[mask]).sum() / sum_weight
            y_pred.append(proba)
        return pd.Series(y_pred, X.index)
    
    def predict(self, X: pd.DataFrame):
        y_pred = self.predict_proba(X)
        return y_pred >= 0.5
        


class MyBaggingClf:
    def __init__(self, estimator = None , n_estimators = 10, max_samples = 1.0, random_state = 42, oob_score = None):
        self.estimator = estimator
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.random_state = random_state
        self.estimators = []
        self.oob_score = oob_score
        self.oob_score_ = None

    def __str__(self):
        return f"MyBaggingClf class: estimator={self.estimator}, n_estimators={self.n_estimators}, max_samples={self.max_samples}, random_state={self.random_state}"

    def fit(self, X: pd.DataFrame, y: pd.Series):
        n = round(self.max_samples * len(X))
        random.seed(self.random_state)
        X_s = []
        y_s = []
        if self.oob_score is not None:
            oob_X_s = []
            oob_y_s = []
            oob_y_pred_s = []
        for i in range(self.n_estimators):
            sample_rows_idx = random.choices(X.index.to_list(), k=n)
            X_hat = X.loc[sample_rows_idx]
            y_hat = y.loc[sample_rows_idx]
            index = pd.Index(range(len(X_hat)))
            X_hat.index = index
            y_hat.index = index
            X_s.append(X_hat)
            y_s.append(y_hat)

            if self.oob_score is not None:
                set_sample_rows_idx = set(sample_rows_idx)
                oob_indexes = X.index[X.index.map(lambda x: x not in set_sample_rows_idx)]
                oob_X = X.loc[oob_indexes]
                oob_y = y.loc[oob_indexes]
                oob_X_s.append(oob_X)
                oob_y_s.append(oob_y)
                

        for i in range(self.n_estimators):
            estimator = copy.deepcopy(self.estimator)
            estimator.fit(X_s[i], y_s[i])
            self.estimators.append(estimator)

            if self.oob_score is not None:
                oob_y_pred = estimator.predict_proba(oob_X_s[i])
                oob_y_pred_s.append(oob_y_pred)

        if self.oob_score is not None:
            oob_y_pred = pd.concat(oob_y_pred_s, axis=1).mean(axis=1)
            list_index_oob = oob_y_pred.index.to_list()
            oob_y_true = y.loc[list_index_oob]

            if self.oob_score == "accuracy":
                metric = accuracy
                
            if self.oob_score == "precision":
                metric = precision
                
            if self.oob_score == "recall":
                metric = recall
                
            if self.oob_score == "f1":
                metric = f1
                
            if self.oob_score == "roc_auc":
                metric = roc_auc

            if self.oob_score != "roc_auc":
                oob_y_pred = oob_y_pred > 0.5
                
            self.oob_score_ = metric(oob_y_true, oob_y_pred)

    def predict(self, X: pd.DataFrame, type):
        y_pred_s = []
        for estimator in self.estimators:
            y_pred = estimator.predict_proba(X)
            y_pred_s.append(y_pred)
        y_pred = pd.concat(y_pred_s, axis=1)
        if type == "mean":
            return y_pred.mean(axis=1) > 0.5
        if type == "vote":
            return (y_pred > 0.5).mean(axis=1) >= 0.5

    def predict_proba(self, X: pd.DataFrame):
        y_pred_s = []
        for estimator in self.estimators:
            y_pred = estimator.predict_proba(X)
            y_pred_s.append(y_pred)
        y_pred = pd.concat(y_pred_s, axis=1)
        return y_pred.mean(axis=1)
        